In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.conf import SparkConf
from pyspark.context import SparkContext

In [2]:
credentials_location = './dbt_demo/service-account.json'

conf = SparkConf() \
    .setMaster('local[*]') \
    .setAppName('test') \
    .set("spark.jars", "./lib/gcs-connector-hadoop3-2.2.5.jar") \
    .set("spark.hadoop.google.cloud.auth.service.account.enable", "true") \
    .set("spark.hadoop.google.cloud.auth.service.account.json.keyfile", credentials_location)

In [3]:
sc = SparkContext(conf=conf)

hadoop_conf = sc._jsc.hadoopConfiguration()

hadoop_conf.set("fs.AbstractFileSystem.gs.impl",  "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFS")
hadoop_conf.set("fs.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem")
hadoop_conf.set("fs.gs.auth.service.account.json.keyfile", credentials_location)
hadoop_conf.set("fs.gs.auth.service.account.enable", "true")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/05 16:50:54 WARN Utils: Your hostname, Osis-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.188 instead (on interface en0)
26/03/05 16:50:54 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
26/03/05 16:50:54 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/05 16:50:56 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [4]:
spark = SparkSession.builder \
    .config(conf=sc.getConf()) \
    .getOrCreate()

In [16]:
import subprocess

result = subprocess.run(
    ['gsutil', 'ls', '-r', 'gs://gcs_spark_week6/pq/'],
    capture_output=True, text=True
)

green_paths = list(set(
    p.strip().rsplit('/', 1)[0] + '/'
    for p in result.stdout.splitlines()
    if 'green' in p and p.endswith('.parquet')
))

df_green = spark.read.parquet(*green_paths)


In [17]:
df_green.count()

2304517

In [ ]:
#All these steps are how to connect your local spark cluster to your google storage